In [9]:
import os, glob, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

CONFIG = {
    'xml_root': r'E:\DATA\Annotations',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'target_label': 'bangla-tesla',
    'batch_size': 32,
    'epochs': 30,
    'lr': 1e-3
}
print(f"✅ S-GAN Config Loaded. Device: {CONFIG['device']}")

✅ S-GAN Config Loaded. Device: cpu


In [10]:
class BanglaTeslaDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45, target_label='bangla-tesla'):
        self.seq_len = obs_len + pred_len
        self.samples = []
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        for xml in tqdm(xml_files, desc="Parsing XMLs"):
            try:
                tree = ET.parse(xml)
                meta = tree.getroot().find('meta').find('original_size')
                img_w, img_h = float(meta.find('width').text), float(meta.find('height').text)
                
                for track in tree.getroot().findall('track'):
                    if track.attrib['label'] != target_label: continue
                    boxes = sorted(track.findall('box'), key=lambda b: int(b.attrib['frame']))
                    data = []
                    for box in boxes:
                        if box.get('outside') == '1': continue
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        data.append([(xtl+xbr)/(2*img_w), (ytl+ybr)/(2*img_h), (xbr-xtl)/img_w, (ybr-ytl)/img_h])
                    
                    data = np.array(data)
                    if len(data) < self.seq_len: continue
                    
                    for i in range(0, len(data) - self.seq_len + 1, 5):
                        self.samples.append({
                            'obs': data[i:i+obs_len],
                            'target': data[i+obs_len:i+obs_len+pred_len],
                            'goal': data[i+obs_len+pred_len-1]
                        })
            except: pass

    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        # FIX: Explicitly use dtype=torch.float32
        return (
            torch.tensor(s['obs'], dtype=torch.float32), 
            torch.tensor(s['target'], dtype=torch.float32), 
            torch.tensor(s['goal'], dtype=torch.float32)
        )

dataset = BanglaTeslaDataset(CONFIG['xml_root'])
train_set, val_set = torch.utils.data.random_split(dataset, [int(0.8*len(dataset)), len(dataset)-int(0.8*len(dataset))])
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)
print(f"✅ Data Ready: {len(train_set)} Train")

Parsing XMLs:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Data Ready: 3410 Train


In [11]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.LSTM(4, 256, batch_first=True)
        self.dec = nn.LSTM(4, 256, batch_first=True)
        self.fc = nn.Linear(256, 4)
        self.noise = nn.Linear(256+16, 256)
    def forward(self, obs):
        _, (h, c) = self.enc(obs)
        z = torch.randn(h.size(1), 16).to(h.device)
        h = self.noise(torch.cat([h.squeeze(0), z], 1)).unsqueeze(0)
        outs = []
        inp = obs[:, -1, :].unsqueeze(1)
        for _ in range(45):
            out, (h, c) = self.dec(inp, (h, c))
            pred = self.fc(out)
            outs.append(pred)
            inp = pred
        return torch.cat(outs, 1)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.LSTM(4, 256, batch_first=True)
        self.cls = nn.Sequential(nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, traj):
        _, (h, _) = self.enc(traj)
        return self.cls(h.squeeze(0))

G = Generator().to(CONFIG['device'])
D = Discriminator().to(CONFIG['device'])
opt_g = optim.Adam(G.parameters(), lr=CONFIG['lr'])
opt_d = optim.Adam(D.parameters(), lr=CONFIG['lr'])
bce, mse = nn.BCELoss(), nn.MSELoss()
print("✅ S-GAN Models Initialized")

✅ S-GAN Models Initialized


In [12]:
print("🚀 Training S-GAN...")

for epoch in range(CONFIG['epochs']):
    G.train(); D.train()
    g_loss_ep, d_loss_ep = 0, 0
    
    # --- TRAIN ---
    for obs, target in tqdm(train_loader, leave=False):
        obs, target = obs.to(CONFIG['device']), target.to(CONFIG['device'])
        batch = obs.size(0)
        
        # Train Discriminator
        opt_d.zero_grad()
        real_traj = torch.cat([obs, target], 1)
        fake_traj = torch.cat([obs, G(obs)], 1)
        l_d = bce(D(real_traj), torch.ones(batch,1).to(CONFIG['device'])) + \
              bce(D(fake_traj.detach()), torch.zeros(batch,1).to(CONFIG['device']))
        l_d.backward()
        opt_d.step()
        
        # Train Generator
        opt_g.zero_grad()
        fake_traj = torch.cat([obs, G(obs)], 1)
        l_g = bce(D(fake_traj), torch.ones(batch,1).to(CONFIG['device'])) + 10*mse(fake_traj[:,15:], target)
        l_g.backward()
        opt_g.step()
        
        g_loss_ep += l_g.item()
        d_loss_ep += l_d.item()

    # --- VALIDATION ---
    G.eval()
    val_l2_ep = 0
    with torch.no_grad():
        for obs, target in val_loader:
            obs, target = obs.to(CONFIG['device']), target.to(CONFIG['device'])
            
            # For GAN Validation, we mostly care about L2 Accuracy (MSE) of the generator
            # Discriminator loss on validation is not very interpretable
            pred_path = G(obs)
            l2 = mse(pred_path, target)
            val_l2_ep += l2.item()
            
    print(f"Epoch {epoch+1} | Train G_Loss: {g_loss_ep/len(train_loader):.4f} | Val L2 Loss: {val_l2_ep/len(val_loader):.6f}")

torch.save(G.state_dict(), "bangla_tesla_sgan.pth")

🚀 Training S-GAN...


  0%|          | 0/107 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 2)